# Notebook 01 — Single Run Walkthrough

Run one simulation for 365 days with fixed seed 42 and produce:
1. Stacked area chart of fleet status with shift change markers
2. Per-aircraft availability heatmap
3. Parts inventory level over time
4. Maintenance queue length over time
5. Summary table of KPIs

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.simulation import FleetSimulation
from src.distributions import weibull_mean
from src.subsystem import SUBSYSTEM_PARAMS

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 120

HOURS_PER_DAY = 24

print('Modules loaded successfully.')

## Weibull Parameter Summary

In [ ]:
print('Subsystem Weibull Parameters & Analytical MTTF')
print('=' * 55)
print(f'{"Subsystem":<14} {"Beta":<8} {"Eta (hrs)":<12} {"MTTF (hrs)":<12}')
print('-' * 55)
for name, params in SUBSYSTEM_PARAMS.items():
    mttf = weibull_mean(params['beta'], params['eta'])
    print(f'{name:<14} {params["beta"]:<8.1f} {params["eta"]:<12.0f} {mttf:<12.1f}')

## Run Simulation

In [ ]:
sim = FleetSimulation(
    fleet_size=12,
    o_level_techs_day=8,
    i_level_techs_day=4,
    d_level_techs=6,
    sortie_interval_hours=48,
)

result = sim.run(
    simulation_days=365,
    surge_start_day=None,
    surge_duration_hours=None,
    random_seed=42,
)

print('Simulation complete.')
print(f'MCR:  {result["mcr"]*100:.1f}%')
print(f'FMCR: {result["fmcr"]*100:.1f}%')
print(f'PAM:  {result["pam_fraction"]*100:.1f}%')
print(f'Total Sorties: {result["total_sorties"]}')
print(f'Total Flight Hours: {result["total_flight_hours"]:.0f}')

## 1. Fleet Status Stacked Area Chart

In [ ]:
kpi_df = result['kpi_dataframe']
time_days = kpi_df['time'] / HOURS_PER_DAY

fig, ax = plt.subplots(figsize=(16, 6))

categories = ['fmc', 'sortie', 'nmc_maintenance', 'pam', 'nmc_depot']
labels = ['FMC', 'On Sortie', 'In Maintenance', 'Awaiting Parts', 'In Depot']
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#8e44ad']

y_stack = np.zeros((len(kpi_df), len(categories)))
for i, cat in enumerate(categories):
    if cat in kpi_df.columns:
        y_stack[:, i] = kpi_df[cat].values

ax.stackplot(time_days, y_stack.T, labels=labels, colors=colors, alpha=0.85)

# Shift change markers (every 12 hours = shift boundary)
for day in range(0, 366, 7):
    ax.axvline(x=day, color='gray', linestyle=':', alpha=0.2, linewidth=0.5)

# Warm-up boundary
ax.axvline(x=30, color='red', linestyle='--', alpha=0.7, label='Warm-up (30d)')

ax.set_xlabel('Simulation Day', fontsize=12)
ax.set_ylabel('Number of Aircraft', fontsize=12)
ax.set_title('Fleet Status Over Time (365 Days, Seed=42)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Per-Aircraft Availability Heatmap

In [ ]:
fleet = result['fleet']
n_weeks = int(365 * HOURS_PER_DAY / (HOURS_PER_DAY * 7))
heatmap_data = np.ones((12, n_weeks))

for i, ac in enumerate(fleet):
    for event in ac.maintenance_events:
        start_week = int(event['start_time'] / (HOURS_PER_DAY * 7))
        end_week = int(event['end_time'] / (HOURS_PER_DAY * 7))
        for w in range(max(0, start_week), min(n_weeks, end_week + 1)):
            heatmap_data[i, w] = 0

fig, ax = plt.subplots(figsize=(16, 5))
im = ax.imshow(heatmap_data, cmap='RdYlGn', aspect='auto',
               interpolation='nearest', vmin=0, vmax=1)
ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Aircraft', fontsize=12)
ax.set_yticks(range(12))
ax.set_yticklabels([f'AC-{i:02d}' for i in range(12)])
ax.set_title('Per-Aircraft Availability (Green=Available, Red=In Maintenance)',
             fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Availability', shrink=0.6)
plt.tight_layout()
plt.show()

## 3. Parts Inventory Level Over Time

In [ ]:
stock_df = pd.DataFrame(result['stock_history'])
stock_df['day'] = stock_df['time'] / HOURS_PER_DAY

from src.inventory import INVENTORY_PARAMS

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
subsystems = ['engine', 'avionics', 'hydraulics', 'airframe']
colors_sub = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for idx, (sub, color) in enumerate(zip(subsystems, colors_sub)):
    ax = axes[idx // 2, idx % 2]
    col_name = f'{sub}_stock'
    if col_name in stock_df.columns:
        ax.plot(stock_df['day'], stock_df[col_name], color=color,
                alpha=0.7, linewidth=0.8)
        # Reorder point line
        r = INVENTORY_PARAMS[sub]['reorder_point']
        ax.axhline(y=r, color='red', linestyle='--', alpha=0.6,
                   label=f'Reorder point (r={r})')
    ax.set_title(f'{sub.capitalize()} Parts', fontweight='bold')
    ax.set_xlabel('Day')
    ax.set_ylabel('Stock Level')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Spare Parts Inventory Levels Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Maintenance Queue Length Over Time

In [ ]:
queue_df = pd.DataFrame(result['queue_history'])
queue_df['day'] = queue_df['time'] / HOURS_PER_DAY

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(queue_df['day'], queue_df['o_level_queue'], label='O-level', color='#3498db', linewidth=0.8)
ax.plot(queue_df['day'], queue_df['i_level_queue'], label='I-level', color='#f39c12', linewidth=0.8)
ax.plot(queue_df['day'], queue_df['d_level_queue'], label='D-level', color='#8e44ad', linewidth=0.8)
ax.set_xlabel('Simulation Day', fontsize=12)
ax.set_ylabel('Queue Length', fontsize=12)
ax.set_title('Maintenance Queue Length Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Summary Table

In [ ]:
# Compute mean maintenance pipeline time
maint_log = result['maintenance_log']
if maint_log:
    mean_pipeline = np.mean([e['total_downtime_hours'] for e in maint_log])
else:
    mean_pipeline = 0

summary = pd.DataFrame({
    'Metric': ['Mission Capable Rate (MCR)', 'Full Mission Capable Rate (FMCR)',
               'PAM Fraction', 'Mean Maintenance Pipeline Time (hours)',
               'Total Sorties'],
    'Value': [f'{result["mcr"]*100:.1f}%', f'{result["fmcr"]*100:.1f}%',
              f'{result["pam_fraction"]*100:.1f}%', f'{mean_pipeline:.1f}',
              f'{result["total_sorties"]}']
})

print('\n' + '=' * 50)
print('  SIMULATION SUMMARY (365 days, seed=42)')
print('=' * 50)
for _, row in summary.iterrows():
    print(f'  {row["Metric"]:<45} {row["Value"]}')
print('=' * 50)
summary